# Comparing Site Definitions: Ion Migration in an FCC Lattice

This tutorial introduces the core concepts of `site-analysis` by working through a simple example: tracking a lithium ion as it migrates between interstitial sites in an FCC lattice. You will define sites using three different approaches — spherical, Voronoi, and polyhedral — and compare how each one assigns the mobile ion during a migration event.

## Prerequisites

This tutorial requires the following packages:

- `site-analysis`
- `matplotlib`

All data in this tutorial is generated synthetically, so no external data files are needed.

## Overview

We will create a 3x3x3 FCC oxygen lattice and a synthetic trajectory of a single lithium ion migrating from an octahedral site, through a tetrahedral site, to another octahedral site. We then analyse this trajectory using three different site definitions and compare the results.

In [1]:
%config InlineBackend.figure_format = 'retina'

## Creating the FCC lattice

First, create the host framework — a 3x3x3 supercell of an FCC oxygen lattice:

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
from pymatgen.core import Lattice, Structure, DummySpecies
from site_analysis import TrajectoryBuilder

a = 4.0  # lattice parameter
lattice = Lattice.cubic(a)
supercell_expansion = [3, 3, 3]
fcc_structure = Structure.from_spacegroup(
    sg='Fm-3m',
    lattice=lattice,
    species=['O'],
    coords=[[0, 0, 0]]
) * supercell_expansion

The FCC structure has two types of interstitial sites:

- **Octahedral sites**: at edge midpoints (0.5, 0.0, 0.0) and body centres (0.5, 0.5, 0.5)
- **Tetrahedral sites**: at positions like (0.25, 0.25, 0.25)

We generate the fractional coordinates for both site types using `pymatgen`'s space group machinery with a `DummySpecies`:

In [3]:
fcc_octahedral_coords = (Structure.from_spacegroup(
    sg='Fm-3m',
    lattice=lattice,
    species=[DummySpecies('Q')],
    coords=[[0.5, 0, 0]]
) * supercell_expansion).frac_coords

fcc_tetrahedral_coords = (Structure.from_spacegroup(
    sg='Fm-3m',
    lattice=lattice,
    species=[DummySpecies('Q')],
    coords=[[0.25, 0.25, 0.25]]
) * supercell_expansion).frac_coords

## Generating a test trajectory

Next, create a synthetic trajectory of a lithium ion migrating from one octahedral site, through a tetrahedral site, to another octahedral site. This mimics a common diffusion mechanism in close-packed structures.

We define three waypoints — one at each site — and use a quadratic spline to interpolate a smooth curved path between them, sampled at 21 evenly spaced frames:

In [4]:
waypoints = np.array([
    [1/3, 2/3, 1/2],      # octahedral site
    [5/12, 7/12, 7/12],   # tetrahedral site
    [1/2, 2/3, 2/3],      # octahedral site
])

t_waypoints = np.array([0.0, 0.5, 1.0])
t = np.linspace(0, 1, 21)

spline = make_interp_spline(t_waypoints, waypoints, k=2)
mobile_ion_coords = spline(t)

md_trajectory = []
for c in mobile_ion_coords:
    structure = fcc_structure.copy().append(species='Li', coords=c)
    md_trajectory.append(structure)

print(f'{len(md_trajectory)} frames')

21 frames


## Analysing with spherical sites

We use the `TrajectoryBuilder` to set up each analysis. The builder configures the structure, mobile species, and site definitions, then builds a `Trajectory` object. Calling `trajectory_from_structures` processes each frame, assigning the mobile ion to a site at each timestep. For more details on the builder pattern, see the [builders guide](https://site-analysis.readthedocs.io/en/latest/guides/builders.html).

Spherical sites are the simplest site type: each site is a sphere defined by a centre position and radius. We first choose radii equal to the inradius of each polyhedron (the radius of the largest sphere that fits inside it):

- Octahedral inradius: r = l / sqrt(6) = 1.155 A
- Tetrahedral inradius: r = l * sqrt(6) / 12 = 0.577 A

For more details, see the [spherical sites guide](https://site-analysis.readthedocs.io/en/latest/guides/spherical_sites.html).

In [5]:
builder = (
    TrajectoryBuilder()
    .with_structure(md_trajectory[0])
    .with_mobile_species("Li")
    .with_spherical_sites(
        centres=fcc_octahedral_coords,
        radii=1.155,
        labels="octahedral"
    ).with_spherical_sites(
        centres=fcc_tetrahedral_coords,
        radii=0.577,
        labels="tetrahedral"
    )
)

trajectory = builder.build()
trajectory.trajectory_from_structures(md_trajectory, progress=True)

spherical_trajectory = trajectory.atoms[0].trajectory
print(spherical_trajectory)

Analysing trajectory:   0%|          | 0/21 [00:00<?, ? steps/s]

[70, 70, 70, 70, 70, 70, None, 283, 283, 283, 283, 283, 283, 283, None, 17, 17, 17, 17, 17, 17]


Notice the `None` values at frames 6 and 14 — the ion is between sites, in regions not covered by any sphere.

### What if we use larger radii?

The inradii leave gaps because inscribed spheres only touch the faces of each polyhedron, leaving the corners uncovered. We can instead use the circumradius of each polyhedron, which is the distance from the centre to the vertices and gives the radius of the smallest sphere that fully encloses it:

- Octahedral circumradius: r = l / sqrt(2) = 2.0 A
- Tetrahedral circumradius: r = l * sqrt(6) / 4 = 1.732 A

For comparison, the distance between neighbouring octahedral and tetrahedral site centres is only sqrt(3) = 1.732 A. The octahedral circumsphere therefore extends well past its tetrahedral neighbour's centre, and the tetrahedral circumsphere reaches the octahedral centre. This means there is substantial overlap between neighbouring sites.

When an atom lies inside multiple spheres simultaneously, `site-analysis` uses a persistence rule: the atom remains assigned to its current site as long as it is still within that site's volume. This avoids spurious transitions from small oscillations, but means the effective boundary between sites depends on which direction the atom is travelling.

In [ ]:
builder = (
    TrajectoryBuilder()
    .with_structure(md_trajectory[0])
    .with_mobile_species("Li")
    .with_spherical_sites(
        centres=fcc_octahedral_coords,
        radii=2.0,
        labels="octahedral"
    ).with_spherical_sites(
        centres=fcc_tetrahedral_coords,
        radii=1.732,
        labels="tetrahedral"
    )
)

trajectory = builder.build()
trajectory.trajectory_from_structures(md_trajectory, progress=True)

spherical_circumradii_trajectory = trajectory.atoms[0].trajectory
print(spherical_circumradii_trajectory)

No `None` values — every frame is assigned. But the persistence rule means the transition timing depends on the direction of travel: the atom stays in its current site until it leaves that sphere, even if it has already entered a neighbouring one.

## Analysing with Voronoi sites

Voronoi sites partition the entire space based on proximity to site centres — every point is assigned to its nearest site centre, so there are no gaps or overlaps. For more details, see the [Voronoi sites guide](https://site-analysis.readthedocs.io/en/latest/guides/voronoi_sites.html).

In [6]:
builder = (TrajectoryBuilder()
    .with_structure(md_trajectory[0])
    .with_mobile_species("Li")
    .with_voronoi_sites(
        centres=fcc_octahedral_coords,
        labels="octahedral"
    ).with_voronoi_sites(
        centres=fcc_tetrahedral_coords,
        labels="tetrahedral"
    )
)

trajectory = builder.build()
trajectory.trajectory_from_structures(md_trajectory, progress=True)

voronoi_trajectory = trajectory.atoms[0].trajectory
print(voronoi_trajectory)

Analysing trajectory:   0%|          | 0/21 [00:00<?, ? steps/s]

[70, 70, 70, 70, 70, 283, 283, 283, 283, 283, 283, 283, 283, 283, 283, 283, 17, 17, 17, 17, 17]


With Voronoi sites the ion is always assigned to a site — no `None` values.

## Analysing with polyhedral sites

Polyhedral sites are defined by coordination polyhedra formed by the host lattice atoms. This requires a reference structure that marks where each site type is located, using dummy atoms. For more details, see the [polyhedral sites guide](https://site-analysis.readthedocs.io/en/latest/guides/polyhedral_sites.html).

In [7]:
reference_structure = fcc_structure.copy()

for frac_coord in fcc_octahedral_coords:
    reference_structure.append("Mg", frac_coord)

for site in fcc_tetrahedral_coords:
    reference_structure.append("Na", site)

builder = (TrajectoryBuilder()
    .with_structure(md_trajectory[0])
    .with_reference_structure(reference_structure)
    .with_mobile_species("Li")
    .with_polyhedral_sites(
        centre_species="Mg",
        vertex_species="O",
        cutoff=3.0,
        n_vertices=6,
        label="octahedral"
    ).with_polyhedral_sites(
        centre_species="Na",
        vertex_species="O",
        cutoff=2.5,
        n_vertices=4,
        label="tetrahedral"
    ).with_site_mapping(mapping_species='O')
          )

trajectory = builder.build()
trajectory.trajectory_from_structures(md_trajectory, progress=True)

polyhedral_trajectory = trajectory.atoms[0].trajectory
print(polyhedral_trajectory)

Analysing trajectory:   0%|          | 0/21 [00:00<?, ? steps/s]

[70, 70, 70, 70, 70, 70, 283, 283, 283, 283, 283, 283, 283, 283, 283, 17, 17, 17, 17, 17, 17]


Like Voronoi sites, polyhedral sites fill space completely through their face-sharing structure, so the ion is always assigned.

## Comparing the results

We can visualise the site assignment for each method to see how they differ:

In [ ]:
site_index = {17: 1, 283: 2, 70: 3, None: 0}

trajectories = {
    'Spherical (inradii)': spherical_trajectory,
    'Spherical (circumradii)': spherical_circumradii_trajectory,
    'Voronoi Sites': voronoi_trajectory,
    'Polyhedral Sites': polyhedral_trajectory,
}

fig, axes = plt.subplots(4, 1, figsize=(5, 8), sharex=True)

for ax, (title, traj) in zip(axes, trajectories.items()):
    ax.plot([site_index[i] for i in traj], 'o--', markersize=3, color='tab:red')
    ax.set_yticks([0, 1, 2, 3])
    ax.set_yticklabels(['unassigned', 'oct', 'tet', 'oct'])
    ax.set_ylim(-0.2, 3.2)
    ax.set_xlim(-0.8, 20.8)
    ax.set_title(title, loc='center')

axes[-1].set_xticks(list(range(0, 21, 5)))
axes[-1].set_xlabel('Frame')
fig.tight_layout()
plt.show()

## Analysis

All four analyses capture the essential migration path: octahedral → tetrahedral → octahedral. The differences lie in how they handle the transitions. For a more detailed discussion of site types and their tradeoffs, see the [sites concept page](https://site-analysis.readthedocs.io/en/latest/concepts/sites.html).

**Spherical sites** are defined by their radii. If we use non-overlapping spheres then we leave spaces between sites. In the example above, the inscribed-sphere radii do not cover the corners of the polyhedra, so the ion appears "unassigned" at the mid-points of transitions (frames 6 and 14). If we increase the sphere radii to fill space, then we eliminate the gaps, but neighbouring spheres now overlap. When the ion enters a region covered by multiple sites, `site-analysis` keeps it assigned to its current site until it leaves that sphere. This means the transition point depends on the direction of travel — the ion leaves the octahedral site at a different frame than it would enter it from the other direction. This persistence rule is deterministic and avoids spurious transitions, but the effective site boundaries are no longer invariant with respect to the direction of ion motion.

**Voronoi sites** partition all space by proximity to site centres, so ions are always assigned. However, the boundaries depend only on the site centre positions and do not reflect the local coordination environment of the host structure.

**Polyhedral sites** are defined by the geometries of coordination polyhedra. Transitions occur when an ion passes through the faces of these polyhedra. This often gives a physically meaningful boundary based on the actual crystal structure. The transition timing differs slightly from the Voronoi approach because, in this case, the shared face between adjacent tetrahedra and octahedra is not at the midpoint between site centres.

## Summary

- **Spherical sites** require choosing both centre positions and radii. If the radii are too small, there will be gaps between sites and unassigned periods during transitions. If the radii are large enough to fill space, sites will overlap and the persistence rule determines the effective boundaries.
- **Voronoi sites** require only centre positions and fill space based on proximity, ensuring unambigious assignment and continuous tracking.
- **Polyhedral sites** can also fill space completely (if we use a set of space-filling polyhedra), but use coordination polyhedra defined by the positions of framework atoms to define boundaries.